In [ ]:
#1

!rm -rf /kaggle/working/buddi

!cp -r /kaggle/input/buddi-kaggle-upload/buddi /kaggle/working/buddi

%cd /kaggle/working/buddi

!ls
!ls essentials/buddi
!ls essentials/body_models/smpl
!ls essentials/body_models/smplx
!ls essentials/body_models/smil

In [ ]:
#2

import torch, sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("STOP: GPU is not enabled. Go to Kaggle Settings → Accelerator → GPU, then restart session.")

In [ ]:
#3

%cd /kaggle/working/buddi

!pip install -q smplx scipy scikit-image loguru omegaconf ipdb einops trimesh opencv-python
!pip install -q mmcv==1.3.9 timm
!pip install -q mmdet==2.25.1
!pip install -q -e third-party/ViTPose/
!pip install -q simple_romp==1.1.3

In [ ]:
#4

%%bash
set -e

cd /kaggle/working/buddi

echo "=== Reinstall clean chumpy ==="
pip uninstall -y chumpy || true
pip install -q chumpy==0.70

echo "=== Patch chumpy for Python 3.12 / NumPy 2 ==="
python - <<'PY'
from pathlib import Path
import site

patch = """import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

import numpy as _np
if not hasattr(_np, 'bool'):
    _np.bool = bool
if not hasattr(_np, 'int'):
    _np.int = int
if not hasattr(_np, 'float'):
    _np.float = float
if not hasattr(_np, 'complex'):
    _np.complex = complex
if not hasattr(_np, 'object'):
    _np.object = object
if not hasattr(_np, 'str'):
    _np.str = str
if not hasattr(_np, 'unicode'):
    _np.unicode = str
"""

found = False
for sp in site.getsitepackages():
    init_file = Path(sp) / "chumpy" / "__init__.py"
    if init_file.exists():
        old = init_file.read_text()
        if "np.unicode = str" not in old:
            init_file.write_text(patch + "\n" + old)
        print("Patched:", init_file)
        found = True

if not found:
    raise FileNotFoundError("Could not find chumpy/__init__.py")
PY

echo "=== Test chumpy ==="
python -c "import chumpy; print('chumpy OK')"

echo "=== Copy SMPL files to where ROMP/BEV expects them ==="
cp essentials/body_models/smpl/SMPL_NEUTRAL.pkl essentials/body_models/SMPL_NEUTRAL.pkl
cp essentials/body_models/smpl/SMPL_NEUTRAL.pkl essentials/body_models/SMPL_MALE.pkl
cp essentials/body_models/smpl/SMPL_NEUTRAL.pkl essentials/body_models/SMPL_FEMALE.pkl

echo "=== Download/copy ROMP extra model files ==="
rm -rf smpl_model_data smpl_model_data.zip

wget -q https://github.com/Arthur151/ROMP/releases/download/V2.0/smpl_model_data.zip -O smpl_model_data.zip
unzip -q smpl_model_data.zip

cp smpl_model_data/J_regressor_extra.npy essentials/body_models/J_regressor_extra.npy
cp smpl_model_data/J_regressor_h36m.npy essentials/body_models/J_regressor_h36m.npy
cp smpl_model_data/smpl_kid_template.npy essentials/body_models/smpl_kid_template.npy

echo "=== Prepare ROMP/BEV body files ==="
romp.prepare_smpl -source_dir=essentials/body_models
bev.prepare_smil -source_dir=essentials/body_models

echo "=== Copy generated files into BUDDI essentials ==="
mkdir -p essentials/body_models/smpla
cp /root/.romp/SMPLA_NEUTRAL.pth essentials/body_models/smpla/SMPLA_NEUTRAL.pth
cp /root/.romp/smil_packed_info.pth essentials/body_models/smil/smil_packed_info.pth

echo "=== Check generated files ==="
echo "===== /root/.romp ====="
ls /root/.romp

echo "===== smpla ====="
ls essentials/body_models/smpla

echo "===== smil ====="
ls essentials/body_models/smil

In [ ]:
#5
%cd /kaggle/working/buddi

import importlib.util
import pathlib

if importlib.util.find_spec("pytorch3d") is not None:
    print("pytorch3d installed OK")
else:
    print("Creating minimal pytorch3d fallback")
    root = pathlib.Path("/kaggle/working/buddi/pytorch3d/transforms")
    root.mkdir(parents=True, exist_ok=True)

    (root.parent / "__init__.py").write_text("")
    (root / "__init__.py").write_text("""
import torch
import torch.nn.functional as F

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = F.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-2)

def matrix_to_rotation_6d(matrix):
    return matrix[..., :2, :].clone().reshape(*matrix.shape[:-2], 6)

def axis_angle_to_matrix(axis_angle):
    angle = torch.norm(axis_angle, dim=-1, keepdim=True).clamp(min=1e-8)
    axis = axis_angle / angle
    x, y, z = axis.unbind(-1)
    ca = torch.cos(angle[..., 0])
    sa = torch.sin(angle[..., 0])
    C = 1 - ca
    xs, ys, zs = x*sa, y*sa, z*sa
    xC, yC, zC = x*C, y*C, z*C
    xyC, yzC, zxC = x*yC, y*zC, z*xC
    m = torch.stack([
        x*xC + ca, xyC - zs, zxC + ys,
        xyC + zs, y*yC + ca, yzC - xs,
        zxC - ys, yzC + xs, z*zC + ca
    ], dim=-1)
    return m.reshape(axis_angle.shape[:-1] + (3, 3))

def matrix_to_axis_angle(matrix):
    cos_angle = ((matrix[..., 0, 0] + matrix[..., 1, 1] + matrix[..., 2, 2]) - 1) / 2
    cos_angle = cos_angle.clamp(-1 + 1e-6, 1 - 1e-6)
    angle = torch.acos(cos_angle)
    rx = matrix[..., 2, 1] - matrix[..., 1, 2]
    ry = matrix[..., 0, 2] - matrix[..., 2, 0]
    rz = matrix[..., 1, 0] - matrix[..., 0, 1]
    axis = torch.stack([rx, ry, rz], dim=-1)
    axis = F.normalize(axis, dim=-1)
    return axis * angle.unsqueeze(-1)
""")
    print("fallback created")

!PYTHONPATH=/kaggle/working/buddi:$PYTHONPATH python -c "from pytorch3d.transforms import rotation_6d_to_matrix; print('pytorch3d transforms OK')"

In [ ]:
#6

%cd /kaggle/working/buddi

import os
from pathlib import Path
import re
import shutil

os.environ["PYTHONPATH"] = "/kaggle/working/buddi:" + os.environ.get("PYTHONPATH", "")

# ============================================================
# 1. Use ViTPose-B checkpoint and patch config to ViTPose-B
# ============================================================

!pip install -q huggingface_hub

!mkdir -p essentials/vitpose
!rm -f essentials/vitpose/wholebody.pth

from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="loreleva/vitpose",
    filename="wholebody/vitpose-b-wholebody.pth",
    local_dir="/kaggle/working/hf_vitpose",
    local_dir_use_symlinks=False
)

shutil.copy(ckpt_path, "essentials/vitpose/wholebody.pth")

size = Path("essentials/vitpose/wholebody.pth").stat().st_size
print("ViTPose-B checkpoint size:", size)

if size < 100_000_000:
    raise RuntimeError("ViTPose-B checkpoint is too small. Download failed.")
else:
    print("ViTPose-B checkpoint looks OK.")

# Find a ViTPose base wholebody config
vitpose_root = Path("/kaggle/working/buddi/third-party/ViTPose")
config_candidates = list(vitpose_root.rglob("*base*wholebody*.py")) + list(vitpose_root.rglob("*b*wholebody*.py"))

print("Possible ViTPose-B configs:")
for c in config_candidates[:20]:
    print(c)

if len(config_candidates) == 0:
    raise FileNotFoundError("Could not find ViTPose-B wholebody config in third-party/ViTPose/configs")

# Prefer 256x192 base wholebody config if available
base_cfg = None
for c in config_candidates:
    name = str(c).lower()
    if "base" in name and "wholebody" in name and "256x192" in name:
        base_cfg = c
        break

if base_cfg is None:
    for c in config_candidates:
        name = str(c).lower()
        if "base" in name and "wholebody" in name:
            base_cfg = c
            break

if base_cfg is None:
    base_cfg = config_candidates[0]

print("Using ViTPose config:", base_cfg)

# Patch BUDDI ViTPose script to use this config and checkpoint
vitpose_file = Path("/kaggle/working/buddi/llib/utils/keypoints/vitpose_model.py")
s = vitpose_file.read_text()

# Replace config paths ending in .py with the selected base config
s = re.sub(
    r"config_file\s*=\s*['\"].*?\.py['\"]",
    f"config_file = '{base_cfg}'",
    s
)

s = re.sub(
    r"config\s*=\s*['\"].*?\.py['\"]",
    f"config = '{base_cfg}'",
    s
)

# Replace checkpoint path if hardcoded
s = re.sub(
    r"checkpoint_file\s*=\s*['\"].*?\.pth['\"]",
    "checkpoint_file = 'essentials/vitpose/wholebody.pth'",
    s
)

s = re.sub(
    r"checkpoint\s*=\s*['\"].*?\.pth['\"]",
    "checkpoint = 'essentials/vitpose/wholebody.pth'",
    s
)

# Patch torch.load for PyTorch 2.6+
patch_code = """
# ---- Kaggle/PyTorch 2.6+ patch: force torch.load(weights_only=False) ----
import torch as _torch_for_patch
_original_torch_load = _torch_for_patch.load
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)
_torch_for_patch.load = _patched_torch_load
# ------------------------------------------------------------------------
"""

if "_patched_torch_load" not in s:
    insert_pos = s.find("class")
    if insert_pos == -1:
        insert_pos = 0
    s = s[:insert_pos] + patch_code + "\n" + s[insert_pos:]

vitpose_file.write_text(s)
print("Patched vitpose_model.py")

# Show important lines for checking
!grep -n "config_file\|checkpoint_file\|wholebody.pth" /kaggle/working/buddi/llib/utils/keypoints/vitpose_model.py | head -20

# ============================================================
# 2. Patch Python 3.12 dataclass issues in BUDDI defaults
# ============================================================

def patch_dataclass_file(path):
    path = Path(path)
    text = path.read_text()
    original = text

    if "from dataclasses import dataclass" in text:
        line = text.split("from dataclasses import dataclass", 1)[1].split("\n", 1)[0]
        if "field" not in line:
            text = text.replace(
                "from dataclasses import dataclass",
                "from dataclasses import dataclass, field"
            )

    def find_matching_paren(src, open_pos):
        depth = 0
        for i in range(open_pos, len(src)):
            if src[i] == "(":
                depth += 1
            elif src[i] == ")":
                depth -= 1
                if depth == 0:
                    return i
        raise ValueError("No matching parenthesis found")

    pattern = re.compile(
        r"(?P<indent>^[ \t]*)(?P<name>\w+)\s*:\s*(?P<class>[A-Z][A-Za-z0-9_]*)\s*=\s*(?P=class)\(",
        re.MULTILINE
    )

    pos = 0
    new_text = ""
    count = 0

    while True:
        m = pattern.search(text, pos)
        if not m:
            new_text += text[pos:]
            break

        line_start = text.rfind("\n", 0, m.start()) + 1
        line_end = text.find("\n", m.start())
        if line_end == -1:
            line_end = len(text)
        line = text[line_start:line_end]

        if "field(default_factory" in line:
            new_text += text[pos:line_end]
            pos = line_end
            continue

        open_pos = text.find("(", m.start())
        close_pos = find_matching_paren(text, open_pos)
        inner = text[open_pos + 1:close_pos]

        replacement = (
            f"{m.group('indent')}{m.group('name')}: {m.group('class')} = "
            f"field(default_factory=lambda: {m.group('class')}({inner}))"
        )

        new_text += text[pos:m.start()] + replacement
        pos = close_pos + 1
        count += 1

    if new_text != original:
        path.write_text(new_text)
        print(f"Patched {count} defaults in {path}")
    else:
        print(f"No changes needed in {path}")

for pyfile in Path("/kaggle/working/buddi/llib/defaults").rglob("*.py"):
    patch_dataclass_file(pyfile)

# ============================================================
# 3. Clean previous demo outputs
# ============================================================

!rm -rf demo/data/vitpose_live
!rm -rf demo/data/bev_live
!rm -rf demo/optimization

print("Cell 6 finished successfully.")

In [ ]:
#7

%cd /kaggle/working/buddi

import os
from pathlib import Path
import shutil

os.environ["PYTHONPATH"] = "/kaggle/working/buddi:" + os.environ.get("PYTHONPATH", "")

# ============================================================
# Use FlickrCI3D test images instead of old demo images
# ============================================================

src_root = Path("/kaggle/input/flickrci3d-test/test/images")

if not src_root.exists():
    raise FileNotFoundError(f"FlickrCI3D images folder not found: {src_root}")

image_exts = {".jpg", ".jpeg", ".png"}
images = sorted([p for p in src_root.rglob("*") if p.suffix.lower() in image_exts])

print("Total FlickrCI3D images found:", len(images))

if len(images) == 0:
    raise RuntimeError("No images found inside FlickrCI3D test/images")

# Clean old demo images and outputs
!rm -rf demo/data/images_live
!rm -rf demo/data/vitpose_live
!rm -rf demo/data/bev_live
!rm -rf demo/optimization

Path("demo/data/images_live").mkdir(parents=True, exist_ok=True)

# Copy more images, not only 3
N_IMAGES = 100

for i, img in enumerate(images[:N_IMAGES]):
    dst = Path("demo/data/images_live") / f"flickr_{i:04d}{img.suffix.lower()}"
    shutil.copy(img, dst)

print(f"Copied {min(N_IMAGES, len(images))} images to demo/data/images_live")

print("\n=== Current input images ===")
!find demo/data/images_live -maxdepth 1 -type f | head -20

# Run demo
print("\n=== Running BUDDI demo on FlickrCI3D images ===")
!PYTHONPATH=/kaggle/working/buddi:$PYTHONPATH bash demo.sh

print("\n=== ViTPose outputs ===")
!ls demo/data/vitpose_live | head -20 || true

print("\n=== BEV outputs ===")
!ls demo/data/bev_live | head -20 || true

print("\n=== Optimization outputs ===")
!find demo/optimization -maxdepth 5 -type f | head -100 || true

In [ ]:
#8

%cd /kaggle/working/buddi

print("=== Current input images ===")
!find demo/data/images_live -maxdepth 1 -type f | head -30

print("\n=== ViTPose outputs ===")
!ls demo/data/vitpose_live | head -30 || true

print("\n=== BEV outputs ===")
!ls demo/data/bev_live | head -30 || true

print("\n=== Optimization outputs ===")
!find demo/optimization -maxdepth 5 -type f | head -100 || true

In [ ]:
from pathlib import Path

result_dir = Path("/kaggle/working/buddi/demo/optimization/buddi_cond_bev_demo_live/fit_buddi_cond_bev_flickrci3ds/results")
image_dir = Path("/kaggle/working/buddi/demo/optimization/buddi_cond_bev_demo_live/fit_buddi_cond_bev_flickrci3ds/images")

print("PKL results:", len(list(result_dir.glob("*.pkl"))))
print("OBJ meshes:", len(list(result_dir.glob("*.obj"))))
print("Visualization images:", len(list(image_dir.glob("*.png"))))

print("\nExample result files:")
for p in sorted(result_dir.glob("*"))[:20]:
    print(p.name)

In [ ]:
from pathlib import Path
import shutil

base_dir = Path("/kaggle/working/buddi/demo/optimization/buddi_cond_bev_demo_live/fit_buddi_cond_bev_flickrci3ds")
backup_dir = Path("/kaggle/working/task3_baseline_outputs")

if backup_dir.exists():
    shutil.rmtree(backup_dir)

shutil.copytree(base_dir, backup_dir)

print("Saved baseline outputs to:", backup_dir)

print("Baseline result files:")
for p in sorted((backup_dir / "results").glob("*")):
    print(p.name)

In [ ]:
from pathlib import Path
from IPython.display import Image, display

image_dir = Path("/kaggle/working/task3_baseline_outputs/images")

imgs = sorted(list(image_dir.glob("*.png")) + list(image_dir.glob("*.jpg")))

print("Visualization images:", len(imgs))

for img in imgs[:10]:
    print(img.name)
    display(Image(filename=str(img)))

In [ ]:
import shutil
from pathlib import Path

zip_path = "/kaggle/working/task3_baseline_outputs"
shutil.make_archive(zip_path, "zip", "/kaggle/working/task3_baseline_outputs")

print("Created:", zip_path + ".zip")

In [ ]:
from pathlib import Path
import shutil

src = Path("/kaggle/working/buddi/llib/methods/hhcs_optimization/main.py")
dst = Path("/kaggle/working/buddi/llib/methods/hhcs_optimization/main_task3_noise.py")

shutil.copy(src, dst)

print("Created:", dst)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/buddi/llib/methods/hhcs_optimization")

keywords = [
    "bev",
    "init",
    "initial",
    "estimates",
    "cam_trans",
    "smpl",
    "translation",
    "global_orient"
]

for pyfile in root.rglob("*.py"):
    text = pyfile.read_text(errors="ignore").lower()
    hits = [k for k in keywords if k in text]
    if "bev" in hits or "init" in hits:
        print("\nFILE:", pyfile)
        for i, line in enumerate(pyfile.read_text(errors="ignore").splitlines(), start=1):
            low = line.lower()
            if any(k in low for k in keywords):
                print(f"{i}: {line[:160]}")

In [ ]:
from pathlib import Path
import numpy as np

bev_dir = Path("/kaggle/working/buddi/demo/data/bev_live")

npz_files = sorted(bev_dir.glob("*.npz"))

print("Number of BEV npz files:", len(npz_files))
print("First files:")
for p in npz_files[:10]:
    print(" -", p.name)

print("\n=== Inspect first BEV npz file ===")
sample = npz_files[0]
print("Sample:", sample)

data = np.load(sample, allow_pickle=True)

for k in data.files:
    arr = data[k]
    try:
        print(k, type(arr), arr.shape, arr.dtype)
    except Exception:
        print(k, type(arr))

In [ ]:
from pathlib import Path
import numpy as np

bev_dir = Path("/kaggle/working/buddi/demo/data/bev_live")
sample = sorted(bev_dir.glob("*.npz"))[0]

print("Sample:", sample)

npz = np.load(sample, allow_pickle=True)
results = npz["results"]

print("results type:", type(results))
print("results shape:", results.shape)

# Convert scalar object array to real Python object
obj = results.item()

print("\nObject type:", type(obj))

if isinstance(obj, dict):
    print("\nDictionary keys:")
    for k, v in obj.items():
        try:
            print(k, type(v), getattr(v, "shape", None), getattr(v, "dtype", None))
        except Exception as e:
            print(k, type(v), "ERROR:", e)
else:
    print("\nObject content:")
    print(obj)

In [ ]:
from pathlib import Path
import numpy as np
import shutil

# Paths
buddi_root = Path("/kaggle/working/buddi")
bev_dir = buddi_root / "demo/data/bev_live"

# Backup clean BEV outputs
clean_backup = Path("/kaggle/working/task3_clean_bev_live")
if clean_backup.exists():
    shutil.rmtree(clean_backup)
shutil.copytree(bev_dir, clean_backup)

print("Backed up clean BEV files to:", clean_backup)
print("Number of clean BEV files:", len(list(clean_backup.glob("*.npz"))))

# Noise settings
noise_levels = {
    "noise_00_clean": {"trans_noise": 0.00, "rot_noise": 0.00},
    "noise_01_mild": {"trans_noise": 0.05, "rot_noise": 0.05},
    "noise_02_medium": {"trans_noise": 0.10, "rot_noise": 0.10},
    "noise_03_hard": {"trans_noise": 0.20, "rot_noise": 0.20},
}

def add_noise_to_bev_npz(src_npz, dst_npz, trans_noise=0.0, rot_noise=0.0, seed=42):
    rng = np.random.default_rng(seed)

    npz = np.load(src_npz, allow_pickle=True)
    results = npz["results"].item()

    noisy = {}

    for k, v in results.items():
        if isinstance(v, np.ndarray):
            noisy[k] = v.copy()
        else:
            noisy[k] = v

    # Add translation noise to 3D body locations
    if "cam_trans" in noisy:
        n_people = noisy["cam_trans"].shape[0]
        delta = rng.normal(0, trans_noise, size=noisy["cam_trans"].shape).astype(np.float32)
        noisy["cam_trans"] = noisy["cam_trans"] + delta

        # Keep mesh/joints consistent with translation change
        if "verts" in noisy:
            noisy["verts"] = noisy["verts"] + delta[:, None, :]
        if "joints" in noisy:
            noisy["joints"] = noisy["joints"] + delta[:, None, :]

    # Add rotation/pose noise to global orientation only
    if "smpl_thetas" in noisy:
        pose_delta = rng.normal(0, rot_noise, size=noisy["smpl_thetas"][:, 0:3].shape).astype(np.float32)
        noisy["smpl_thetas"][:, 0:3] = noisy["smpl_thetas"][:, 0:3] + pose_delta

    # Add small noise to camera scale/translation parameters if available
    if "cam" in noisy:
        cam_delta = rng.normal(0, trans_noise * 0.2, size=noisy["cam"].shape).astype(np.float32)
        noisy["cam"] = noisy["cam"] + cam_delta

    np.savez_compressed(dst_npz, results=noisy)

# Create noisy BEV folders
for level_name, cfg in noise_levels.items():
    out_dir = Path(f"/kaggle/working/task3_bev_{level_name}")
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, src in enumerate(sorted(clean_backup.glob("*.npz"))):
        dst = out_dir / src.name
        add_noise_to_bev_npz(
            src,
            dst,
            trans_noise=cfg["trans_noise"],
            rot_noise=cfg["rot_noise"],
            seed=1000 + i
        )

    print(level_name, "created:", len(list(out_dir.glob("*.npz"))), "files")

In [ ]:
from pathlib import Path
import numpy as np

clean_file = sorted(Path("/kaggle/working/task3_clean_bev_live").glob("*.npz"))[0]
noisy_file = sorted(Path("/kaggle/working/task3_bev_noise_03_hard").glob("*.npz"))[0]

clean = np.load(clean_file, allow_pickle=True)["results"].item()
noisy = np.load(noisy_file, allow_pickle=True)["results"].item()

print("File:", clean_file.name)

print("Clean cam_trans:")
print(clean["cam_trans"])

print("\nNoisy cam_trans:")
print(noisy["cam_trans"])

print("\nDifference:")
print(noisy["cam_trans"] - clean["cam_trans"])

print("\nClean global orientation:")
print(clean["smpl_thetas"][:, 0:3])

print("\nNoisy global orientation:")
print(noisy["smpl_thetas"][:, 0:3])

In [ ]:
from pathlib import Path
import shutil
import subprocess
import os

buddi_root = Path("/kaggle/working/buddi")
os.environ["PYTHONPATH"] = str(buddi_root) + ":" + os.environ.get("PYTHONPATH", "")

# Folders created in the previous noise cell
noise_dirs = {
    "noise_00_clean": Path("/kaggle/working/task3_bev_noise_00_clean"),
    "noise_01_mild": Path("/kaggle/working/task3_bev_noise_01_mild"),
    "noise_02_medium": Path("/kaggle/working/task3_bev_noise_02_medium"),
    "noise_03_hard": Path("/kaggle/working/task3_bev_noise_03_hard"),
}

# Make an optimization-only script from demo.sh
demo_sh = buddi_root / "demo.sh"
opt_only_sh = buddi_root / "run_buddi_optimization_only.sh"

lines = demo_sh.read_text().splitlines()

start_idx = None
for i, line in enumerate(lines):
    if "Running Optimization with BUDDI" in line:
        start_idx = i
        break

if start_idx is None:
    raise RuntimeError("Could not find 'Running Optimization with BUDDI' inside demo.sh")

opt_lines = [
    "#!/bin/bash",
    "set -e",
    "cd /kaggle/working/buddi",
    "export PYTHONPATH=/kaggle/working/buddi:$PYTHONPATH",
] + lines[start_idx:]

opt_only_sh.write_text("\n".join(opt_lines) + "\n")
opt_only_sh.chmod(0o755)

print("Created optimization-only script:", opt_only_sh)

# Output folder for Task 3
task3_out_root = Path("/kaggle/working/task3_noise_results")
if task3_out_root.exists():
    shutil.rmtree(task3_out_root)
task3_out_root.mkdir(parents=True, exist_ok=True)

# Make sure ViTPose outputs exist
vitpose_dir = buddi_root / "demo/data/vitpose_live"
if not vitpose_dir.exists():
    raise RuntimeError("demo/data/vitpose_live does not exist. Run the working demo cell first.")

# Run optimization for each noise level
for level_name, noisy_bev_dir in noise_dirs.items():
    print("\n" + "=" * 80)
    print("Running BUDDI optimization for:", level_name)
    print("=" * 80)

    if not noisy_bev_dir.exists():
        raise FileNotFoundError(f"Missing noisy BEV folder: {noisy_bev_dir}")

    # Replace demo/data/bev_live with noisy BEV
    current_bev = buddi_root / "demo/data/bev_live"
    if current_bev.exists():
        shutil.rmtree(current_bev)
    shutil.copytree(noisy_bev_dir, current_bev)

    # Clean optimization output before each run
    opt_dir = buddi_root / "demo/optimization"
    if opt_dir.exists():
        shutil.rmtree(opt_dir)

    # Run optimization only
    result = subprocess.run(
        ["bash", str(opt_only_sh)],
        cwd=str(buddi_root),
        env=os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout[-4000:])  # show last part of log

    if result.returncode != 0:
        raise RuntimeError(f"Optimization failed for {level_name}")

    # Copy optimization output
    produced_dir = buddi_root / "demo/optimization/buddi_cond_bev_demo_live/fit_buddi_cond_bev_flickrci3ds"

    if not produced_dir.exists():
        raise RuntimeError(f"Expected output folder not found for {level_name}: {produced_dir}")

    dst = task3_out_root / level_name
    shutil.copytree(produced_dir, dst)

    n_pkl = len(list((dst / "results").glob("*.pkl")))
    n_obj = len(list((dst / "results").glob("*.obj")))
    n_img = len(list((dst / "images").glob("*"))) if (dst / "images").exists() else 0

    print(f"Saved {level_name} outputs to:", dst)
    print("PKL:", n_pkl, "OBJ:", n_obj, "Images:", n_img)

print("\nAll noise experiments finished.")
print("Results saved in:", task3_out_root)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/task3_noise_results")

for level_dir in sorted(root.iterdir()):
    if not level_dir.is_dir():
        continue

    result_dir = level_dir / "results"
    image_dir = level_dir / "images"

    print("\n", level_dir.name)
    print("PKL results:", len(list(result_dir.glob("*.pkl"))) if result_dir.exists() else 0)
    print("OBJ meshes:", len(list(result_dir.glob("*.obj"))) if result_dir.exists() else 0)
    print("Visualization images:", len(list(image_dir.glob("*"))) if image_dir.exists() else 0)

In [ ]:
import shutil

zip_path = "/kaggle/working/task3_noise_results"
shutil.make_archive(zip_path, "zip", "/kaggle/working/task3_noise_results")

print("Created:", zip_path + ".zip")

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

root = Path("/kaggle/working/task3_noise_results")

levels = [
    "noise_00_clean",
    "noise_01_mild",
    "noise_02_medium",
    "noise_03_hard",
]

# Find common visualization image names
image_sets = []
for level in levels:
    img_dir = root / level / "images"
    imgs = sorted([p.name for p in img_dir.glob("*.png")])
    image_sets.append(set(imgs))

common_imgs = sorted(set.intersection(*image_sets))

print("Common visualization images:", common_imgs)

for img_name in common_imgs:
    fig, axes = plt.subplots(1, len(levels), figsize=(5 * len(levels), 5))
    
    for ax, level in zip(axes, levels):
        img_path = root / level / "images" / img_name
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(level)
        ax.axis("off")
    
    plt.suptitle(img_name)
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/kaggle/working/task3_noise_results")

rows = []

for level_dir in sorted(root.iterdir()):
    if not level_dir.is_dir():
        continue
    
    result_dir = level_dir / "results"
    image_dir = level_dir / "images"
    
    rows.append({
        "Noise level": level_dir.name,
        "PKL results": len(list(result_dir.glob("*.pkl"))) if result_dir.exists() else 0,
        "OBJ meshes": len(list(result_dir.glob("*.obj"))) if result_dir.exists() else 0,
        "Visualization images": len(list(image_dir.glob("*.png"))) if image_dir.exists() else 0,
    })

df = pd.DataFrame(rows)
df